[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_01_CICD_Pipeline.ipynb)

**Run order (do these once, then repeat step 3 after every work session)**

| Step | What | Notebook |
|---|---|---|
| 0 | Create a GitHub fine-grained token for this repo only (Contents + Workflows read/write, Actions read). In Colab: key icon (Secrets) > add `GITHUB_KEY-Rodic-infra-Colab` > turn on Notebook access. Never paste it in a cell or in chat. | none |
| 1 | Set up CI/CD once: writes `.github/workflows`, dry-runs lint and tests, pushes, reads run status. | [01 CICD](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_01_CICD_Pipeline.ipynb) |
| 2 | First sync as a dry run: set `DRY_RUN = True` in the config cell and check the staged file list. | [00 Sync](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync_v2.ipynb) |
| 3 | Real sync: `DRY_RUN = False`. Run top to bottom; the last cell pushes. | [00 Sync](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync_v2.ipynb) |

Within each notebook, run cells top to bottom (Runtime > Run all is fine). The last cell is always the push.

# Rodic_01_CICD_Pipeline
Creates and pushes a GitHub Actions CI/CD pipeline for `Rodic-infra-Colab`, checks it locally first, then reads the run status back into Colab.

**Pipeline design (starter, tighten later)**
- **CI** on every push/PR to `main`: install, lint (syntax and undefined names only, so it starts green), secret-pattern check, pytest.
- **CD**: Streamlit Cloud already redeploys `v2/app_v2.py` automatically when `main` changes, so no deploy step is needed there. Pushing a `v*` tag builds a zip and creates a GitHub Release.
- Not included yet: model-accuracy gates and edge-device (Jetson) tests, since real checkpoints and hardware are not wired in.

Token needs Contents and Workflows read/write plus Actions read. Reuse the `GITHUB_KEY-Rodic-infra-Colab` secret from notebook 00.

### Config and authentication helpers
Same helper block as notebook 00 so this notebook runs on its own.

In [1]:
import os, re, json, base64, shutil, subprocess, pathlib, datetime
from google.colab import userdata

# ---------- CONFIG: edit only this block ----------
REPO         = "Subhash-Pal/Rodic-infra-Colab"
BRANCH       = "main"
VERSION_DIR  = "v4"                 # folder inside the repo that receives your work (Vx/ rule)
WORK_DIR     = "/content/work"      # folder in THIS Colab runtime whose files get pushed
NOTEBOOK_PATH = ""                  # optional: Drive path of the Colab you are editing, e.g.
                                    # "/content/drive/MyDrive/Colab Notebooks/Rodic_Edge_CV.ipynb"
RUN_TESTS    = True                 # run pytest before pushing, if tests exist
DRY_RUN      = False                # True = stage + show diff, no commit/push
MAX_FILE_MB  = 50
# --------------------------------------------------

REPO_DIR = "/content/_repo"
def _pick_secret(*names):
    for n in names:
        try:
            v = userdata.get(n)
        except Exception:
            v = None
        if v:
            return n, v
    raise RuntimeError("No GitHub token secret found. Add Colab Secret GITHUB_KEY-Rodic-infra-Colab and turn on Notebook access.")
TOKEN_NAME, TOKEN = _pick_secret("GITHUB_KEY-Rodic-infra-Colab", "GITHUB_TOKEN", "GITHUB_TOKEN-gemini-resilient")
_kind = "fine-grained" if TOKEN.startswith("github_pat_") else "classic" if TOKEN.startswith("ghp_") else "unknown type"
print(f"Using secret: {TOKEN_NAME} ({_kind})")
try:
    import json as _j, urllib.request as _u
    _r = _u.urlopen(_u.Request("https://api.github.com/user", headers={"Authorization": f"Bearer {TOKEN}"}), timeout=15)
    print("Token belongs to:", _j.load(_r)["login"], "| classic scopes:", _r.headers.get("X-OAuth-Scopes") or "n/a (fine-grained: check repo access on GitHub)")
except Exception as e:
    print("Token check failed:", e)
_B64 = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
AUTH = ["-c", f"http.extraheader=AUTHORIZATION: basic {_B64}"]

def _mask(t): return t.replace(TOKEN, "***").replace(_B64, "***")

def git(*args, cwd=REPO_DIR, auth=False, check=True):
    cmd = ["git"] + (AUTH if auth else []) + list(args)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = _mask(r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out
print("Running: 01-cicd v0.4 (pushes whenever local is ahead of GitHub, verifies remote)")


Using secret: GITHUB_KEY-Rodic-infra-Colab (fine-grained)
Token belongs to: Subhash-Pal | classic scopes: n/a (fine-grained: check repo access on GitHub)
Running: 01-cicd v0.4 (pushes whenever local is ahead of GitHub, verifies remote)


### Clone or update the repo

In [2]:
if not os.path.isdir(f"{REPO_DIR}/.git"):
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR, cwd="/content", auth=True)
else:
    git("fetch", "origin", BRANCH, auth=True)
    git("checkout", BRANCH)
    git("pull", "--rebase", "origin", BRANCH, auth=True)
git("config", "user.name", "Subhash Pal (Colab)")
git("config", "user.email", "colab-sync@users.noreply.github.com")
print(git("log", "--oneline", "-5"))

4af1b44 notebooks 00: read secret GITHUB_KEY-Rodic-infra-Colab first, real version stamp v0.5; docs and recovery log
a84c701 notebooks 00/01: read secret GITHUB_KEY-Rodic-infra-Colab first; update docs and recovery log
c0febb7 notebooks 00/01: prefer secret GITHUB_TOKEN, print token owner and type, clearer 403 hint; log the error
e3e5a2a notebooks/01: push when local is ahead of origin, verify remote, add PUSH FAILED hint and version stamp; log the error
1b3733b Created using Colab


### Write CI and release workflow files
Creates `.github/workflows/ci.yml` (lint, secret check, pytest) and `release.yml` (zip + GitHub Release on `v*` tags).

In [3]:
wf = pathlib.Path(REPO_DIR) / ".github" / "workflows"
wf.mkdir(parents=True, exist_ok=True)

(wf / "ci.yml").write_text(r"""name: CI
on:
  push:
    branches: [main]
  pull_request:
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
      - name: Install
        run: |
          pip install -r v2/requirements.txt pytest pyyaml scikit-image scipy ruff
      - name: Lint (syntax and undefined names only)
        run: ruff check --select E9,F63,F7,F82 .
      - name: Secret pattern check
        run: |
          if grep -RIEn --exclude-dir=.git 'AIza[0-9A-Za-z_-]{35}|ghp_[A-Za-z0-9]{36}|sk-ant-[A-Za-z0-9_-]{20,}' . ; then
            echo "Possible secret committed"; exit 1; fi
      - name: Tests
        run: pytest -q
""")
(wf / "release.yml").write_text(r"""name: Release
on:
  push:
    tags: ["v*"]
permissions:
  contents: write
jobs:
  release:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Package
        run: |
          zip -r rodic-${{ github.ref_name }}.zip v2 v3 README.md -x "*/__pycache__/*"
      - name: Create GitHub Release
        uses: softprops/action-gh-release@v2
        with:
          files: rodic-${{ github.ref_name }}.zip
          generate_release_notes: true
""")
print("Wrote:", *sorted(p.name for p in wf.iterdir()))

Wrote: ci.yml release.yml


### Local dry-run of CI
Runs lint and pytest here with the same commands the workflow uses, so the first GitHub run does not start red.

In [4]:
# Run the same checks the CI will run, before pushing.
subprocess.run("pip -q install ruff pytest pyyaml scikit-image scipy", shell=True)
if os.path.exists(f"{REPO_DIR}/v2/requirements.txt"):
    subprocess.run(f"pip -q install -r {REPO_DIR}/v2/requirements.txt", shell=True)
lint = subprocess.run("ruff check --select E9,F63,F7,F82 .", shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print("LINT:", "OK" if lint.returncode == 0 else "FAIL\n" + lint.stdout[-1500:])
t = subprocess.run("python -m pytest -q", shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print("TESTS:", "OK" if t.returncode == 0 else "FAIL")
print(t.stdout[-1500:])
LOCAL_OK = (lint.returncode == 0 and t.returncode == 0)

LINT: OK
TESTS: OK
........................................................................ [ 85%]
............                                                             [100%]
84 passed in 9.91s



### FINAL CELL: commit and push the workflows
Pushes only `.github/workflows/`. Blocked if the local dry-run failed.

In [5]:
if not LOCAL_OK:
    print("BLOCKED: local CI dry-run failed. Fix first, otherwise the pipeline would start red.")
else:
    git("add", ".github/workflows")
    if git("status", "--short"):
        git("commit", "-m", "ci: add CI (lint, secret check, pytest) and tag-based release workflows")
    git("fetch", "origin", BRANCH, auth=True)
    ahead = int(git("rev-list", "--count", f"origin/{BRANCH}..HEAD"))
    if ahead == 0:
        print("Nothing to push: origin already has everything in this clone.")
    else:
        print(f"{ahead} local commit(s) not on GitHub yet - pushing.")
        git("pull", "--rebase", "origin", BRANCH, auth=True)
        try:
            git("push", "origin", BRANCH, auth=True)
        except RuntimeError as e:
            print("PUSH FAILED:\n" + str(e))
            print("\n403 'Permission denied' = the token has no WRITE access to THIS repo (often a token made for another repo).")
            print("Fix: GitHub > Settings > Developer settings > fine-grained token: Only select repositories = Rodic-infra-Colab,")
            print("Contents + Workflows = Read and write, Actions = Read. Save it as Colab Secret GITHUB_KEY-Rodic-infra-Colab (Notebook access on).")
            print("If the error mentions 'workflow', the Workflows permission is the missing piece. Then re-run from the config cell.")
        else:
            print("Pushed.")
    # verify on the remote, not on the local clone
    git("fetch", "origin", BRANCH, auth=True)
    remote = git("ls-tree", "--name-only", f"origin/{BRANCH}", ".github/workflows/", check=False)
    print("On GitHub now:", remote if remote else "NO .github/workflows - not published yet")


1 local commit(s) not on GitHub yet - pushing.
Pushed.
On GitHub now: .github/workflows/ci.yml
.github/workflows/release.yml


### Check GitHub Actions run status
Polls the Actions API and prints the latest runs with links.

In [6]:
import time, urllib.request
def latest_runs(n=3):
    req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/actions/runs?per_page={n}",
        headers={"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"})
    return json.load(urllib.request.urlopen(req))["workflow_runs"]

for _ in range(12):                      # poll up to ~2 minutes
    runs = latest_runs()
    if runs and runs[0]["status"] == "completed": break
    time.sleep(10)
for r in runs:
    print(f'{r["name"]:<10} {r["status"]:<12} {r["conclusion"]}  {r["head_sha"][:7]}  {r["html_url"]}')

CI         completed    success  3fc728b  https://github.com/Subhash-Pal/Rodic-infra-Colab/actions/runs/44260932882


### Optional: tag a release
Set `TAG` (e.g. `v2.1.0`) to trigger the release workflow.

In [7]:
# Optional: cut a release. Set TAG and run. This triggers release.yml (zip + GitHub Release).
TAG = ""        # e.g. "v2.1.0"
if TAG:
    git("tag", TAG)
    git("push", "origin", TAG, auth=True)
    print("Tag pushed:", TAG)
else:
    print("TAG empty - skipped.")

TAG empty - skipped.
